# 03 — Sentiment Model Training

Baseline: **TF-IDF + Logistic Regression** on `data/reviews.csv`. Saves `app/models/sentiment_model.pkl` + `app/models/vectorizer.pkl`, loaded by `nlp_service.py`.

Swap in a fine-tuned DistilBERT later (stretch goal) without changing the API — `nlp_service.analyze_sentiment()` is the only contract the rest of the app depends on.

In [ ]:
import re
import string
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

## 1. Load data
Replace `data/reviews.csv` with the Kaggle 'Women's E-Commerce Clothing Reviews' dataset (or similar) for real training — same `review,sentiment` column schema.

In [ ]:
df = pd.read_csv('../data/reviews.csv')
print(df.shape)
df.head()

In [ ]:
df['sentiment'].value_counts()

## 2. Clean text
Mirrors `nlp_service.clean_text()` so training-time and inference-time preprocessing stay in sync.

In [ ]:
import nltk
try:
    from nltk.corpus import stopwords
    STOPWORDS = set(stopwords.words('english'))
except LookupError:
    nltk.download('stopwords', quiet=True)
    from nltk.corpus import stopwords
    STOPWORDS = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', ' ', text)
    tokens = [t for t in text.split() if t not in STOPWORDS]
    return ' '.join(tokens)

df['clean_review'] = df['review'].apply(clean_text)
df[['review', 'clean_review']].head()

## 3. Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_review'], df['sentiment'],
    test_size=0.2, random_state=42, stratify=df['sentiment']
)
print('Train:', len(X_train), '| Test:', len(X_test))

## 4. TF-IDF vectorization

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
X_train_vec.shape

## 5. Train Logistic Regression baseline

In [ ]:
clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X_train_vec, y_train)

## 6. Evaluate

In [ ]:
y_pred = clf.predict(X_test_vec)
print(classification_report(y_test, y_pred))
print('Confusion matrix:\n', confusion_matrix(y_test, y_pred))

## 7. (Optional) Try Linear SVM instead
Often stronger on sparse TF-IDF features — compare against Logistic Regression before picking a final model.

In [ ]:
from sklearn.svm import LinearSVC
svm_clf = LinearSVC(class_weight='balanced')
svm_clf.fit(X_train_vec, y_train)
svm_pred = svm_clf.predict(X_test_vec)
print(classification_report(y_test, svm_pred))

## 8. Save the final model + vectorizer
Saved to `app/models/` — matches the paths `nlp_service.py` expects. Pick whichever of `clf` / `svm_clf` scored better above.

In [ ]:
import os
os.makedirs('../app/models', exist_ok=True)
joblib.dump(clf, '../app/models/sentiment_model.pkl')
joblib.dump(vectorizer, '../app/models/vectorizer.pkl')
print('Saved sentiment_model.pkl and vectorizer.pkl to ../app/models/')